# RSNA knee: ROI classification + hybrid multitask models, inference

One streaming pass per GPU (`src_bis/infer/pipeline.py`, a copy of `src/infer/pipeline.py` extended with the hybrid models); the studies are split between the GPUs:

1. **Read** (DataLoader workers, the only CPU work): DICOM headers of every slice (order, geometry, Laterality tag), and the pixels of the <= 64 slices kept.
2. **Preprocess** (GPU): percentile normalisation, the 384 px cache, and the 32 x 128 x 128 volume, in the training left-right convention.
3. **Side** (GPU): left / right for *every* series, since a study can hold both knees. The model is used, not the DICOM tag.
4. **ROIs**:
   - series flipped to a right knee, then the ROI U-Net gives one box per class (ACL, MCL, menisci, the three OA compartments, Baker's), each with a confidence;
   - weak boxes are replaced by registration from the same knee's other series;
   - axial boxes are registered from the sagittal / coronal ones.
5. **Crops** (GPU), each model's own crop recipe:
   - ROI crops (8 strided slices);
   - OA compartment crops that stop at the intercondylar midline;
   - one capsule crop per series (the union of the ROI boxes) for effusion and synovitis;
   - a joint crop for contusion and fracture.
6. **Models and recipe**: the weights' `recipe.json` lists, per finding, the crop models to use; several models are combined by rank average over the test studies, and an entry can subtract another finding's score (`minus`). Findings the recipe does not cover stay at 0.5. The recipe and the models are printed below.

float16 autocast throughout. Nothing is written to disk except the per-shard CSVs.

**Hybrid models** (`src_bis/mt3d/`, weights in `<WEIGHTS>/mt3d/<name>/`): they reuse the same caches, sides and consensus boxes (nothing is read, flipped or segmented twice).
- One joint crop per sagittal / coronal series: the union of its 8 ROI boxes + 10%, 32 x 256 x 256, cut once from the GPU cache and shared by every hybrid model.
- A study's series go through each model 4 at a time (sagittal and coronal interleaved), as in validation; the study prediction is the mean over its groups.
- 4 models: MOAKS-pretrained ConvNeXt-tiny x 2 seeds and HGNetV2-B2 x 2 seeds, fullfit.
- `recipe.json` (default): two more members in every ROI finding's rank average, next to the `dataset_2_moaks` models: hybrid ConvNeXt and hybrid HGNetV2, each the mean rank of its 2 seeds (a nested list), so the hybrids weigh 2 / (n + 2). Validation: 0.949 / 0.951 (58 / holdout) vs 0.957 / 0.945 for `dataset_2_moaks`.
- Alternatives: `recipe_flat4.json` (each hybrid model one member), `recipe_all8.json` (the 4 hybrids as one member), `recipe_acl_moa.json` (the same, on ACL and medial OA only).
- Autocast float16 (T4). Validated end to end from the DICOMs: the 4-model hybrid ensemble scores 0.9412 / 0.9517 (58 / holdout) through this pipeline vs 0.9410 / 0.9515 in validation.

## Imports

In [ ]:
import os
import sys
import json
import time

import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    # Code and wheels come from attached datasets; the competition image lacks
    # dynamic_network_architectures (the ROI U-Net) and the DICOM decoders.
    # src_bis needs its own code dataset (src_bis/ = src/ + mt3d/ + the extended infer/pipeline.py).
    !cp -r /kaggle/input/datasets/theoviel/rsna-knee-code-bis/src_bis ./
    !pip install --quiet --no-index --find-links /kaggle/input/datasets/theoviel/rsna-knee-wheels dynamic-network-architectures pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm
    SRC = os.path.abspath("./src_bis")
else:
    SRC = os.path.abspath("../src_bis")

# The pipeline runs as `<this python> -m infer.pipeline` with src_bis/ on the path.
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, SRC)
print("src:", SRC, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      "| CPUs:", os.cpu_count())

## Parameters

In [ ]:
if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"
    WEIGHTS = "/kaggle/input/datasets/theoviel/rsna-knee-weights-2-moaks-hybrid/"   # output/bis/dataset_2_moaks_hybrid
    OUT_DIR = "/tmp/knee/"
else:
    DATA_DIR = "../input/"
    WEIGHTS = "../output/bis/dataset_2_moaks_hybrid/"
    OUT_DIR = "../output/inference_bis/"
# WEIGHTS: side/ and roi/ run folders, crops/<run>/ and mt3d/<name>/ (args.json + ckpt_final.pt), recipes
RECIPE = os.path.join(WEIGHTS, "recipe.json")   # 1 hybrid member per backbone on all 8 findings; or recipe_flat4 / _all8 / _acl_moa.json
AMP = "fp16"                                     # hybrid models' autocast (T4: fp16)

SERIES_DIR = "test_series"                       # <SERIES_DIR>/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
SERIES_CSV = os.path.join(DATA_DIR, "test_series.csv")   # series -> study, plane (folders walked if absent)
GPUS = list(range(torch.cuda.device_count()))    # one process per GPU
WORKERS = max(1, os.cpu_count() // len(GPUS))    # DICOM reading processes per GPU
LIMIT = None                                     # first N studies only (debugging)

def flat(models):
    return [m for x in models for m in (flat(x) if isinstance(x, list) else [x])]

recipe = json.load(open(RECIPE))
models = {m for v in recipe.values() for m in flat(v if isinstance(v, list) else v["models"])}
for m in models:
    assert os.path.exists(os.path.join(WEIGHTS, "crops", m, "ckpt_final.pt")) or \
        os.path.exists(os.path.join(WEIGHTS, "mt3d", m, "ckpt_final.pt")), f"missing weights: {m}"
for w in ("side", "roi"):
    assert os.path.exists(os.path.join(WEIGHTS, w, "ckpt_final.pt")), f"missing weights: {w}"
os.makedirs(OUT_DIR, exist_ok=True)
DATA_DIR, WEIGHTS, OUT_DIR, RECIPE = map(os.path.abspath, (DATA_DIR, WEIGHTS, OUT_DIR, RECIPE))
print("recipe:", recipe)

## Run

In [ ]:
t0 = time.time()
cmd = (f"{sys.executable} -m infer.pipeline --data_dir {DATA_DIR} --series_dir {SERIES_DIR} --series_csv {SERIES_CSV}"
       f" --weights {WEIGHTS} --recipe {RECIPE} --amp {AMP} --out_dir {OUT_DIR} --submission submission.csv"
       f" --gpus {','.join(map(str, GPUS))} --workers {WORKERS}" + (f" --limit {LIMIT}" if LIMIT else ""))
print(cmd)
!{cmd}
print(f"{time.time() - t0:.0f}s")

## Outputs

In [ ]:
shards = sorted(d for d in os.listdir(OUT_DIR) if d.startswith("shard") and os.path.exists(os.path.join(OUT_DIR, d, "series.csv")))
series = pd.concat([pd.read_csv(os.path.join(OUT_DIR, d, "series.csv")) for d in shards], ignore_index=True)
rois = pd.concat([pd.read_csv(os.path.join(OUT_DIR, d, "rois.csv")) for d in shards], ignore_index=True)
print(series.plane.value_counts().to_dict(), "| failed:", int((series.status != "ok").sum()),
      "| sides:", series.side.value_counts().to_dict())
print(len(rois), "boxes:", rois.plane.value_counts().to_dict())
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()